In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

may_2026kaggle_assignment_3_path = kagglehub.competition_download('may-2026kaggle-assignment-3')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# The Mushroom Dataset is a classic machine learning classification benchmark used to predict whether a wild mushroom is Edible (e) or Poisonous (p) based on its physical traits.

**Key Stats**
* Size: 8,124 samples.
* Features: 24 features (Cap shape, Gill color, Odor, Habitat, etc.).
* Target: Binary classification (Edible vs. Poisonous).
* Data Type: Features are a mix of numbers, characters and strings.

In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/may-2026kaggle-assignment-3/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/may-2026kaggle-assignment-3/test.csv')

In [ ]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
train_df.head()

In [ ]:
test_df.head()

# Rubric 1: Identify data types of different columns

In [ ]:
train_df.dtypes

In [ ]:
print("Numerical columns in training data:")
print(train_df.select_dtypes(include=np.number).columns.tolist())

print("\nCategorical columns in training data:")
print(train_df.select_dtypes(exclude=np.number).columns.tolist())

In [ ]:
test_df.dtypes

In [ ]:
num_cols = train_df.select_dtypes(include='number').columns
cat_cols = train_df.select_dtypes(exclude='number').columns

In [ ]:
print("Numerical columns in testing data:")
num_cols

In [ ]:
print("\nCategorical columns in testing data:")
cat_cols

# Rubric 2: Present descriptive statistics of numerical columns

In [ ]:
train_df.describe()

# Rubric 3: Identify and handle the missing values

In [ ]:
train_df.isnull().sum()

In [ ]:
print("Total missing cells:", int(train_df.isnull().sum().sum()))

**Handling strategy:**
* Numerical features: median imputation
* Categorical features: most-frequent imputation

In [ ]:
# fill missing numerical values with median
# median as it is less affected by extreme values than the mean

for col in num_cols:
    train_df[col] = train_df[col].fillna(train_df[col].median())
    test_df[col] = test_df[col].fillna(train_df[col].median())

# replacing with training data median as we should use information from the training data to decide how to preprocess the test data

In [ ]:
# fill missing categorical values with mode
for col in cat_cols:
    if col != 'class':
        #class being the target variable should not be affected
        train_df[col] = train_df[col].fillna(train_df[col].mode()[0])
        test_df[col] = test_df[col].fillna(train_df[col].mode()[0])

In [ ]:
train_df.isnull().sum()

# Rubric 4: Identify and handle duplicates

In [ ]:
train_df.duplicated().sum()

No duplicates, no handling

# Rubric 5: Identify and handle outliers

In [ ]:
for col in num_cols:
    Q1 = train_df[col].quantile(0.25)
    Q3 = train_df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((train_df[col] < lower) | (train_df[col] > upper)).sum()

    print(col, "outliers:", outliers)

print("Outliers were checked and retained because the numerical values are discrete and are valid observations.")

# Rubric 6: Present at least three visualizations and provide insights for the same

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# VISUALIZATION 1 — TARGET DISTRIBUTION

target_column = "class"

plt.figure(figsize = (6 , 4))
train_df[target_column].value_counts().plot(kind = "bar")
plt.title("Distribution of Mushroom Classes")
plt.xlabel("Class")
plt.ylabel("Number of Samples")
plt.xticks(rotation = 0)
plt.tight_layout()
plt.show()

print(
    "Insight: The target distribution can be inspected to determine whether "
    "there is a severe class imbalance."
)

In [ ]:
train_df['class'].value_counts()

In [ ]:
# VISUALIZATION 2 — ODOR VS CLASS
if "odor" in train_df.columns:
    odor_table = pd.crosstab(train_df["odor"], train_df[target_column])

    odor_table.plot(kind = "bar", figsize=(9, 5))
    plt.title("Odor vs Mushroom Class")
    plt.xlabel("Odor")
    plt.ylabel("Number of Samples")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    print(
        "Insight: Odor categories show strong differences in class distribution, "
        "making odor an important predictive feature."
    )
else:
    print("The 'odor' column is not available.")


In [ ]:
# VISUALIZATION 3 — HABITAT VS CLASS
if "habitat" in train_df.columns:
    habitat_table = pd.crosstab(train_df["habitat"], train_df[target_column])

    habitat_table.plot(kind = "bar", figsize=(9, 5))
    plt.title("Habitat vs Mushroom Class")
    plt.xlabel("Habitat")
    plt.ylabel("Number of Samples")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    print(
        "Insight: Different habitats have different class distributions, "
        "so habitat contributes useful information to classification."
    )
else:
    print("The 'habitat' column is not available.")


# Rubric 7: Scale Numerical features and Encode Categorical features

In [ ]:
print("Numerical columns:")
print(num_cols)

print("\nCategorical columns:")
print(cat_cols)

* Numerical features are scaled using StandardScaler, while categorical features are converted into numerical form using OneHotEncoder.
* This ensures that the machine learning models can work with all the features appropriately.

In [ ]:
# train_df.head(2)

In [ ]:
X = train_df.drop('class', axis=1)
y = train_df['class']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
x_train_num_cols = X_train.select_dtypes(include='number').columns
x_train_cat_cols = X_train.select_dtypes(exclude='number').columns

print("Numerical:", x_train_num_cols.tolist())
print("Categorical:", x_train_cat_cols.tolist())

In [ ]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Fit the scaler only on training data
X_train[x_train_num_cols] = scaler.fit_transform(
    X_train[x_train_num_cols]
)

# Use the same scaler on test data
X_test[x_train_num_cols] = scaler.transform(
    X_test[x_train_num_cols]
)

In [ ]:
# One-hot encode categorical columns
X_train = pd.get_dummies(
    X_train,
    columns = x_train_cat_cols
)

X_test = pd.get_dummies(
    X_test,
    columns = x_train_cat_cols
)

# Make sure train and test have exactly the same columns
X_test = X_test.reindex(
    columns = X_train.columns,
    fill_value = 0
)

The pandas.get_dummies() function converts categorical variables into dummy/indicator variables, a process commonly known as one-hot encoding. It transforms a column with text or categories into multiple binary columns containing True/False or 1/0, making the data ready for machine learning algorithms.

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

display(X_train.head())

In [ ]:
X_train.head()

In [ ]:
X_test.head()

In [ ]:
X_train.dtypes

# Rubric 8: Model Building (at least 7)

Importing the neccessary models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

from sklearn.metrics import accuracy_score, classification_report

1. Logistic Regression
   

In [ ]:
# Logistic Regression
logistic_model = LogisticRegression(max_iter=1000)

logistic_model.fit(X_train, y_train)

y_pred = logistic_model.predict(X_test)

print("Logistic Regression Accuracy:",accuracy_score(y_test, y_pred))

2. KNN

In [ ]:
# K-Nearest Neighbors
knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train, y_train)

y_pred = knn_model.predict(X_test)

print("KNN Accuracy:", accuracy_score(y_test, y_pred))

3. Decision Trees

In [ ]:
# Decision Tree
tree_model = DecisionTreeClassifier(random_state=42)

tree_model.fit(X_train, y_train)

y_pred = tree_model.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred))

4. Random Forest Classifier


In [ ]:
# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred))

5. Extra Trees

In [ ]:
# Extra Trees
extra_model = ExtraTreesClassifier(
    n_estimators=100,
    random_state=42
)

extra_model.fit(X_train, y_train)

y_pred = extra_model.predict(X_test)

print("Extra Trees Accuracy:", accuracy_score(y_test, y_pred))

6. Gradient Boosting

In [ ]:
# Gradient Boosting
gb_model = GradientBoostingClassifier(
    random_state=42
)

gb_model.fit(X_train, y_train)

y_pred = gb_model.predict(X_test)

print("Gradient Boosting Accuracy:",
      accuracy_score(y_test, y_pred))

7. Adaboost

In [ ]:
# AdaBoost
ada_model = AdaBoostClassifier(
    random_state=42
)

ada_model.fit(X_train, y_train)

y_pred = ada_model.predict(X_test)

print("AdaBoost Accuracy:",
      accuracy_score(y_test, y_pred))

**Comparing all the models**

In [ ]:
# Compare all 7 models

models = {
    "Logistic Regression": logistic_model,
    "KNN": knn_model,
    "Decision Tree": tree_model,
    "Random Forest": rf_model,
    "Extra Trees": extra_model,
    "Gradient Boosting": gb_model,
    "AdaBoost": ada_model
}

In [ ]:
results = []

for name, model in models.items():
    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Accuracy",
    ascending=False
)

display(results_df)

# Rubric 9: Hyperparameter Tuning on any 3 of the models

The rubric requires tuning at least three models.
We tune:
- Gradient Boosting
- Random Forest
- AdaBoost
`GridSearchCV` uses the training split only. The final validation set remains untouched until evaluation.

In [ ]:
from sklearn.model_selection import GridSearchCV

## Gradient Boosting — Hyperparameter Tuning

In [ ]:
# Create the Gradient Boosting model
gb_model_tuned = GradientBoostingClassifier(random_state=42)

# Parameters we want to test
gb_params = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [2, 3, 4]
}

# 5-fold cross-validation
gb_grid = GridSearchCV(
    gb_model,
    gb_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Train the model
gb_grid.fit(X_train, y_train)

# Display the best parameters
print("Best parameters:")
print(gb_grid.best_params_)

print("\nBest CV accuracy:")
print(gb_grid.best_score_)

# Test the best model on our test/validation set
gb_best = gb_grid.best_estimator_

y_pred_gb = gb_best.predict(X_test)

print("\nTest accuracy:")
print(accuracy_score(y_test, y_pred_gb))

## Random Forest — Hyperparameter Tuning

In [ ]:
# Create the Random Forest model
rf_model_tuned = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

# Parameters we want to test
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

#5-fold cross-validation
rf_grid = GridSearchCV(
    rf_model,
    rf_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Train the models
rf_grid.fit(X_train, y_train)

# Display the best parameters
print("Best parameters:")
print(rf_grid.best_params_)

print("\nBest CV accuracy:")
print(rf_grid.best_score_)

# Test the best model
rf_best = rf_grid.best_estimator_

y_pred_rf = rf_best.predict(X_test)

print("\nTest accuracy:")
print(accuracy_score(y_test, y_pred_rf))

## AdaBoost — Hyperparameter Tuning

In [ ]:
# Create the AdaBoost model
ada_model_tuned = AdaBoostClassifier(random_state=42)

# Parameters we want to test
ada_params = {
    'n_estimators': [50, 100, 200, 300],
    'learning_rate': [0.05, 0.1, 0.5, 1.0]
}

# Try all combinations using 5-fold cross-validation
ada_grid = GridSearchCV(
    ada_model,
    ada_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Train the models
ada_grid.fit(X_train, y_train)

# Display the best parameters
print("Best parameters:")
print(ada_grid.best_params_)

print("\nBest CV accuracy:")
print(ada_grid.best_score_)

# Test the best model
ada_best = ada_grid.best_estimator_

y_pred_ada = ada_best.predict(X_test)

print("\nTest accuracy:")
print(accuracy_score(y_test, y_pred_ada))

## Comparing the tuned models

In [ ]:
# Compare the tuned models

tuned_results = pd.DataFrame({
    'Model': [
        'Gradient Boosting',
        'Random Forest',
        'AdaBoost'
    ],
    'Accuracy': [
        accuracy_score(y_test, y_pred_gb),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_ada)
    ]
})

tuned_results = tuned_results.sort_values(
    'Accuracy',
    ascending=False
)

display(tuned_results)

# Rubric 10: Comparison of model performances

In [ ]:
# MODEL PERFORMANCE COMPARISON
comparison = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'KNN',
        'Decision Tree',
        'Random Forest',
        'Extra Trees',
        'Gradient Boosting',
        'AdaBoost',
        'Tuned Gradient Boosting',
        'Tuned Random Forest',
        'Tuned AdaBoost'
    ],

    'Accuracy': [
        accuracy_score(y_test, logistic_model.predict(X_test)),
        accuracy_score(y_test, knn_model.predict(X_test)),
        accuracy_score(y_test, tree_model.predict(X_test)),
        accuracy_score(y_test, rf_model.predict(X_test)),
        accuracy_score(y_test, extra_model.predict(X_test)),
        accuracy_score(y_test, gb_model.predict(X_test)),
        accuracy_score(y_test, ada_model.predict(X_test)),
        accuracy_score(y_test, y_pred_gb),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_ada)
    ],

    'Type': [
        'Baseline',
        'Baseline',
        'Baseline',
        'Baseline',
        'Baseline',
        'Baseline',
        'Baseline',
        'Tuned',
        'Tuned',
        'Tuned'
    ]
})

# Sort models from highest to lowest accuracy
comparison = comparison.sort_values(
    'Accuracy',
    ascending = False
).reset_index(drop = True)

display(comparison)

Finding the best performing model amongst all the models

In [ ]:
# Find the best performing model
best_model = comparison.iloc[0]

print("Best model:", best_model['Model'])
print("Best accuracy:", best_model['Accuracy'])

In [ ]:
# Use the tuned Gradient Boosting model as the final model

final_model = logistic_model

In [ ]:
# Make a copy of the actual Kaggle test dataset
test_features = test_df.copy()

# Scale numerical columns using the scaler
# that was already fitted on X_train

test_features[x_train_num_cols] = scaler.transform(
    test_features[x_train_num_cols]
)

In [ ]:
# One-hot encode the categorical columns

test_features = pd.get_dummies(
    test_features,
    columns=x_train_cat_cols
)


test_features = test_features.reindex(
    columns = X_train.columns,
    fill_value=0
)

In [ ]:
# Generate predictions for the Kaggle test dataset

test_predictions = final_model.predict(test_features)
print("Number of predictions:", len(test_predictions))

In [ ]:
# Create the submission dataframe

submission = pd.DataFrame({
    'ID': test_df['ID'],
    'class': test_predictions
})

# Save the submission file
submission.to_csv(
    'submission.csv',
    index=False
)
print("submission.csv created successfully!")
display(submission.head())

In [ ]:
# Final checks before submitting to Kaggle

print("Submission shape:", submission.shape)
print("Test data shape:", test_df.shape)

print("\nPrediction distribution:")
print(submission['class'].value_counts())